# Tetris-Multiplayer-RL — Colab setup

Bootstrap a Colab Linux x86_64 environment to:

1. clone the repo
2. install build deps (cmake, g++, python-dev, pybind11, torch, numpy)
3. build the headless `tetris_py` pybind11 module **without raylib** (`-DTETRIS_BUILD_GAME=OFF -DTETRIS_BUILD_PY=ON`)
4. drop the resulting `.so` into `python/sim/`
5. smoke-test the import and check the cross-platform state-hash parity gate

After this notebook completes, you can `from sim import SimGame` and `from common.models import TetrisPolicyNet` from any other Colab notebook in this runtime, and the same checkpoint format works on your local Windows netbot.

## 1. Clone the repo

Replace `<USER>` with your GitHub user (or use your fork URL).

In [ ]:
REPO_URL = 'https://github.com/<USER>/Tetris-Multiplayer-RL.git'

import os
if not os.path.exists('Tetris-Multiplayer-RL'):
    !git clone $REPO_URL
%cd Tetris-Multiplayer-RL
!git pull

## 2. Install build dependencies

Colab already has `cmake`, `g++`, and `python3-dev`, but pin them anyway so this notebook works on a cold runtime.

In [ ]:
!apt-get install -qq cmake g++ python3-dev > /dev/null
!pip install -q pybind11 numpy torch

## 3. Build the pybind11 module (no raylib)

`-DTETRIS_BUILD_GAME=OFF` skips the raylib executable so we don't need to apt-install raylib at all. `-DTETRIS_BUILD_PY=ON` enables the `tetris_py` target. We pass pybind11's CMake dir explicitly so the find_package picks up the pip-installed copy.

In [ ]:
import subprocess, shutil, glob

PYBIND11_DIR = subprocess.check_output(['python', '-m', 'pybind11', '--cmakedir']).decode().strip()
print('pybind11 cmake dir:', PYBIND11_DIR)

!mkdir -p build
!cd build && cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DTETRIS_BUILD_GAME=OFF \
    -DTETRIS_BUILD_PY=ON \
    -DTETRIS_BUILD_TEST=ON \
    -Dpybind11_DIR=$PYBIND11_DIR
!cmake --build build -j --target tetris_py sim_hash_dump

# Drop the freshly built .so next to python/sim/__init__.py so `from sim import SimGame` finds it.
for so in glob.glob('build/tetris_py*.so'):
    print('copying', so, '-> python/sim/')
    shutil.copy(so, 'python/sim/')

## 4. Sanity import

Adds `python/` to `sys.path` and imports the wrapper. Prints the initial state hash for a known seed — this is the value that should match the Windows local build (cross-platform determinism gate).

In [ ]:
import sys
sys.path.insert(0, 'python')

from sim import SimGame, Placement
g = SimGame(42)
print('initial state_hash:', hex(g.state_hash()))
print('current piece id  :', g.current_block_id())
print('next piece id     :', g.next_block_id())
print('legal placements  :', len(g.legal_placements()))
for p in g.legal_placements()[:5]:
    print('   ', p)

## 5. Run the C++ determinism dump

`build/sim_hash_dump` is the C++ test driver. Capture its stdout and write it to `python/tests/_sim_hash_dump.txt` — that's the file `test_determinism_crossplatform.py` parses to compare against the Python replay.

Repeat this exact step on your Windows machine and `diff` the two files. They MUST be byte-identical; if they're not, you have a cross-platform determinism bug to fix before any RL training is meaningful.

In [ ]:
!build/sim_hash_dump > python/tests/_sim_hash_dump.txt
!head -40 python/tests/_sim_hash_dump.txt

## 6. Verify the common/ shared layer imports

If this cell runs without error then the architecture / observation / checkpoint code is wired up and a Colab-trained checkpoint will load on your local Windows netbot.

In [ ]:
import torch
from common.models import TetrisPolicyNet
from common.obs import build_observation
from common.action_mask import legal_mask
from common.checkpoint import save_checkpoint, load_checkpoint

model = TetrisPolicyNet()
obs = build_observation(SimGame(42))
batched = {k: v.unsqueeze(0) for k, v in obs.items()}
logits, value = model(**batched)
print('policy logits shape:', tuple(logits.shape))
print('value shape       :', tuple(value.shape))

save_checkpoint(model, '/tmp/toy.pt', extra={'training_steps': 0})
loaded = load_checkpoint('/tmp/toy.pt')
print('checkpoint round-trip OK')